<p align = "center" draggable=”false” ><img src="https://github.com/AI-Maker-Space/LLM-Dev-101/assets/37101144/d1343317-fa2f-41e1-8af1-1dbb18399719" 
     width="200px"
     height="auto"/>
</p>

<h1 align="center" id="heading">OpenAI Agents SDK - AIM</h1>

In this notebook, we'll go over some of the key features of the OpenAI Agents SDK - as explored through a notebook-ified version of their [Research Bot](https://github.com/openai/openai-agents-python/tree/main/examples/research_bot).

In [1]:
### You don't need to run this cell if you're running this notebook locally. 

#pip install -qU openai-agents

API Key:

In [2]:
import os 
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass()

Nest Async:

In [3]:
import nest_asyncio
nest_asyncio.apply()

## Agents

As may be expected, the primary thing we'll do in the Agents SDK is construct Agents!

Agents are constructed with a few basic properties:

- A prompt, which OpenAI is using the language "instruction" for, that determines the behaviour or goal of the Agent
- A model, the "brain" of the Agent

They also typically include an additional property: 

- Tool(s) that equip the Agent with things it can use to get stuff done

### Task 1: Create Planner Agent

Let's start by creating our "Planner Agent" - which will come up with the initial set of search terms that should answer a query provided by the user. 



In [4]:
from pydantic import BaseModel
from agents import Agent

PLANNER_PROMPT = (
    "You are a helpful research assistant. Given a query, come up with a set of web searches to perform" 
    "to best answer the query. Output between 5 and 20 terms to query for."
)

Next, we'll define the data models that our Planner Agent will use to structure its output. We'll create:

1. `WebSearchItem` - A model for individual search items, containing the search query and reasoning
2. `WebSearchPlan` - A container model that holds a list of search items

These Pydantic models will help ensure our agent returns structured data that we can easily process.


In [5]:
class WebSearchItem(BaseModel):
    reason: str
    "Your reasoning for why this search is important to the query."

    query: str
    "The search term to use for the web search."

class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem]
    """A list of web searches to perform to best answer the query."""

Now we'll create our Planner Agent using the Agent class from the OpenAI Agents SDK. This agent will use the instructions defined in `PLANNER_PROMPT` and will output structured data in the form of our WebSearchPlan model. We're using the GPT-4o model for this agent to ensure high-quality search term generation.

> NOTE: When we provide an `output_type` - the model will return a [structured response](https://platform.openai.com/docs/guides/structured-outputs?api-mode=responses).


In [6]:
planner_agent = Agent(
    name="PlannerAgent",
    instructions=PLANNER_PROMPT,
    model="gpt-4.1",
    output_type=WebSearchPlan,
)

#### ❓Question #1:

Why is it important to provide a structured response template? (As in: Why are structured outputs helpful/preferred in Agentic workflows?)

##### ✅ Answer:
Easy Parsing (avoids random parsing)
Better reliable outcomes
Ease for error handling and predictable output template



### Task 2: Create Search Agent

Now we'll create our Search Agent, which will be responsible for executing web searches based on the terms generated by the Planner Agent. This agent will take each search query, perform a web search using the `WebSearchTool`, and then summarize the results in a concise format.

> NOTE: We are using the `WebSearchTool`, a hosted tool that can be used as part of an `OpenAIResponsesModel` as outlined in the [documentation](https://openai.github.io/openai-agents-python/tools/). This is based on the tools available through OpenAI's new [Responses API](https://openai.com/index/new-tools-for-building-agents/).

The `SEARCH_PROMPT` below instructs the agent to create brief, focused summaries of search results. These summaries are designed to be 2-3 paragraphs, under 300 words, and capture only the essential information without unnecessary details. The goal is to provide the Writer Agent with clear, distilled information that can be efficiently synthesized into the final report.


In [7]:
SEARCH_PROMPT = (
    "You are a research assistant. Given a search term, you search the web for that term and"
    "produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300"
    "words. Capture the main points. Write succinctly, no need to have complete sentences or good"
    "grammar. This will be consumed by someone synthesizing a report, so its vital you capture the"
    "essence and ignore any fluff. Do not include any additional commentary other than the summary"
    "itself."
)

Now we'll create our Search Agent using the Agent class from the OpenAI Agents SDK. This agent will use the instructions defined in `SEARCH_PROMPT` and will utilize the `WebSearchTool` to perform web searches. We're configuring it with `tool_choice="required"` to ensure it always uses the search tool when processing requests.

> NOTE: We can, as demonstrated, indicate how we want our model to use tools. You can read more about that at the bottom of the page [here](https://openai.github.io/openai-agents-python/agents/)

In [8]:
from agents import WebSearchTool
from agents.model_settings import ModelSettings

search_agent = Agent(
    name="Search agent",
    instructions=SEARCH_PROMPT,
    tools=[WebSearchTool()],
    model_settings=ModelSettings(tool_choice="required"),
)

#### ❓ Question #2: 

What other tools are supported in OpenAI's Responses API?

##### ✅ Answer:

From openai doc

The WebSearchTool lets an agent search the web.
The FileSearchTool allows retrieving information from your OpenAI Vector Stores.
The ComputerTool allows automating computer use tasks.
The CodeInterpreterTool lets the LLM execute code in a sandboxed environment.
The HostedMCPTool exposes a remote MCP server's tools to the model.
The ImageGenerationTool generates images from a prompt.
The LocalShellTool runs shell commands on your machine.


### Task 3: Create Writer Agent

Finally, we'll create our Writer Agent, which will synthesize all the research findings into a comprehensive report. This agent takes the original query and the research summaries from the Search Agent, then produces a structured report with follow-up questions.

The Writer Agent will:
1. Create an outline for the report structure
2. Generate a detailed markdown report (5-10 pages)
3. Provide follow-up questions for further research

We'll define the prompt for this agent in the next cell. This prompt will instruct the Writer Agent on how to synthesize research findings into a comprehensive report with follow-up questions.

In [9]:
WRITER_PROMPT = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research "
    "assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words.\n"
    "For the follow-up questions, provide exactly 5 unique questions that would help extend "
    "this research. Do not repeat questions."
)

In [10]:
WRITER_PROMPT = ( "You are a senior Cricket Analyst tasked with writing a cohesive report for a research query. "
        "You will be provided with the original query, and some initial statistical and performance research done by a research " "assistant.\n"
        "You should first come up with an outline for the report that describes the structure and " "flow of the report, focusing on team strategy, player performance metrics, and potential match outcomes/trends. Then, generate the report and return that as your final output.\n" 
        "The final output should be in markdown format, and it should be a detailed analytical deep-dive report. Aim " "for 5-10 pages of content, at least 1000 words.\n" 
        "For the follow-up questions, provide exactly 5 unique questions that would help extend " 
        "this research, focusing on strategic cricketing analysis or future player/team development. Do not repeat questions." )

#### 🏗️ Activity #1: 

This prompt is quite generic - modify this prompt to produce a report that is more personalized to either your personal preference, or more appropriate for a specific use case (eg. law domain research)

Answer
Modified to cater to a Cricket analyst!! 

Now we'll create our Writer Agent using the Agent class from the OpenAI Agents SDK. This agent will synthesize all the research findings into a comprehensive report. We're configuring it with the `ReportData` output type to structure the response with a short summary, markdown report, and follow-up questions.

In [11]:
class ReportData(BaseModel):
    short_summary: str
    """A short 2-3 sentence summary of the findings."""

    markdown_report: str
    """The final report"""

    follow_up_questions: list[str]
    """Suggested topics to research further"""

Now we'll define our Writer Agent using the Agent class from the OpenAI Agents SDK. This agent will take the original query and research summaries, then synthesize them into a comprehensive report with follow-up questions. We've defined a custom output type called `ReportData` that structures the response with a short summary, markdown report, and follow-up questions.

In [12]:
writer_agent = Agent(
    name="WriterAgent",
    instructions=WRITER_PROMPT,
    model="o3-mini",
    output_type=ReportData,
)

#### ❓ Question #3: 

Why are we electing to use a reasoning model for writing our report?

##### ✅ Answer:
Because the agent will need to analyze the prompt, reason out the research, structure the content and then respond ... it isnt just generation but a lot of reasoning is needed to synthesis the right output

## Task 4: Create Utility Classes 

We'll define utility classes to help with displaying progress and managing the research workflow. The Printer class below will provide real-time updates on the research process.


The Printer class provides real-time progress updates during the research process. It uses Rich's Live display to show dynamic content with spinners for in-progress items and checkmarks for completed tasks. The class maintains a dictionary of items with their completion status and can selectively hide checkmarks for specific items. This creates a clean, interactive console experience that keeps the user informed about the current state of the research workflow.

In [17]:
from typing import Any

from rich.console import Console, Group
from rich.live import Live
from rich.spinner import Spinner

class Printer:
    def __init__(self, console: Console):
        self.live = Live(console=console)
        self.items: dict[str, tuple[str, bool]] = {}
        self.hide_done_ids: set[str] = set()
        self.live.start()

    def end(self) -> None:
        self.live.stop()

    def hide_done_checkmark(self, item_id: str) -> None:
        self.hide_done_ids.add(item_id)

    def update_item(
        self, item_id: str, content: str, is_done: bool = False, hide_checkmark: bool = False
    ) -> None:
        self.items[item_id] = (content, is_done)
        if hide_checkmark:
            self.hide_done_ids.add(item_id)
        self.flush()

    def mark_item_done(self, item_id: str) -> None:
        self.items[item_id] = (self.items[item_id][0], True)
        self.flush()

    def flush(self) -> None:
        renderables: list[Any] = []
        for item_id, (content, is_done) in self.items.items():
            if is_done:
                prefix = "✅ " if item_id not in self.hide_done_ids else ""
                renderables.append(prefix + content)
            else:
                renderables.append(Spinner("dots", text=content))
        self.live.update(Group(*renderables))

Let's create a ResearchManager class that will orchestrate the research process. This class will:
1. Plan searches based on the query
2. Perform those searches to gather information
3. Write a comprehensive report based on the gathered information
4. Display progress using our Printer class


In [18]:
from __future__ import annotations

import asyncio
import time

from agents import Runner, custom_span, gen_trace_id, trace

class ResearchManager:
    def __init__(self):
        self.console = Console()
        self.printer = Printer(self.console)

    async def run(self, query: str) -> None:
        trace_id = gen_trace_id()
        with trace("Research trace", trace_id=trace_id):
            self.printer.update_item(
                "trace_id",
                f"View trace: https://platform.openai.com/traces/trace?trace_id={trace_id}",
                is_done=True,
                hide_checkmark=True,
            )

            self.printer.update_item(
                "starting",
                "Starting research...",
                is_done=True,
                hide_checkmark=True,
            )
            search_plan = await self._plan_searches(query)
            search_results = await self._perform_searches(search_plan)
            report = await self._write_report(query, search_results)

            final_report = f"Report summary\n\n{report.short_summary}"
            self.printer.update_item("final_report", final_report, is_done=True)

            self.printer.end()

        print("\n\n=====REPORT=====\n\n")
        print(f"Report: {report.markdown_report}")
        print("\n\n=====FOLLOW UP QUESTIONS=====\n\n")
        unique_questions = []
        seen = set()
        
        for question in report.follow_up_questions:
            if question not in seen:
                unique_questions.append(question)
                seen.add(question)
        
        for i, question in enumerate(unique_questions, 1):
            print(f"{i}. {question}")

    async def _plan_searches(self, query: str) -> WebSearchPlan:
        self.printer.update_item("planning", "Planning searches...")
        result = await Runner.run(
            planner_agent,
            f"Query: {query}",
        )
        self.printer.update_item(
            "planning",
            f"Will perform {len(result.final_output.searches)} searches",
            is_done=True,
        )
        return result.final_output_as(WebSearchPlan)

    async def _perform_searches(self, search_plan: WebSearchPlan) -> list[str]:
        with custom_span("Search the web"):
            self.printer.update_item("searching", "Searching...")
            num_completed = 0
            max_concurrent = 5
            results = []
            
            for i in range(0, len(search_plan.searches), max_concurrent):
                batch = search_plan.searches[i:i+max_concurrent]
                tasks = [asyncio.create_task(self._search(item)) for item in batch]
                
                for task in asyncio.as_completed(tasks):
                    try:
                        result = await task
                        if result is not None:
                            results.append(result)
                    except Exception as e:
                        print(f"Search error: {e}")
                        
                    num_completed += 1
                    self.printer.update_item(
                        "searching", f"Searching... {num_completed}/{len(search_plan.searches)} completed"
                    )
            
            self.printer.mark_item_done("searching")
            return results

    async def _search(self, item: WebSearchItem) -> str | None:
        input = f"Search term: {item.query}\nReason for searching: {item.reason}"
        try:
            result = await Runner.run(
                search_agent,
                input,
            )
            return str(result.final_output)
        except Exception as e:
            print(f"Error searching for '{item.query}': {e}")
            return None

    async def _write_report(self, query: str, search_results: list[str]) -> ReportData:
        self.printer.update_item("writing", "Thinking about report...")
        input = f"Original query: {query}\nSummarized search results: {search_results}"
        
        result = Runner.run_streamed(
            writer_agent,
            input,
        )
        
        update_messages = [
            "Thinking about report...",
            "Planning report structure...",
            "Writing outline...",
            "Creating sections...",
            "Cleaning up formatting...",
            "Finalizing report...",
            "Finishing report...",
        ]

        last_update = time.time()
        next_message = 0
        
        async for event in result.stream_events():
            if time.time() - last_update > 5 and next_message < len(update_messages):
                self.printer.update_item("writing", update_messages[next_message])
                next_message += 1
                last_update = time.time()

        self.printer.mark_item_done("writing")
        return result.final_output_as(ReportData)

#### 🏗️ Activity #2:

Convert the above flow into a flowchart style image (software of your choosing, but if you're not sure which to use try [Excallidraw](https://excalidraw.com/)) that outlines how the different Agents interact with each other. 

> HINT: Cursor's AI (CMD+L or CTRL+L on Windows) would be a helpful way to get a basic diagram that you can add more detail to!


Here's the flowchart showing how the different Agents interact with each other in the research workflow:

![Agent Flow Diagram](AgentFlow.png)

## Task 5: Running Our Agent

Now let's run our agent! The main function below will prompt the user for a research topic, then pass that query to our ResearchManager to handle the entire research process. The ResearchManager will: 

1. Break down the query into search items
2. Search for information on each item
3. Write a comprehensive report based on the search results

Let's see it in action!

In [19]:
async def main() -> None:
    query = input("What would you like to research? ")
    await ResearchManager().run(query)

In [20]:
asyncio.run(main())

Output()



=====REPORT=====


Report: # Comparative Analysis of Sunil Gavaskar and Sachin Tendulkar: A Deep Dive into Batting Greatness

## Introduction

The debate over the greatest batsman in Indian cricket is as enduring as the game itself. This report delves into a comprehensive comparison of two cricketing legends: Sunil Gavaskar and Sachin Tendulkar. While Gavaskar paved the way with his technical precision and resilience against fearsome fast bowling in an era of uncovered pitches, Tendulkar redefined batting through his innovation and consistency over a record-breaking 24-year career. The research query, "Is Sunil Gavaskar a great batsman or Sachin a great batsman?" provides the framework to explore these contrasting yet parallel legacies.

## Historical Context and Era Differences

### Sunil Gavaskar’s Era

Sunil Gavaskar made his debut in 1971, a period marked by challenging pitch conditions, lack of modern protective gear, and the absence of advanced batting technology. The challenge



=====REPORT=====


Report: # Comparative Analysis of Sunil Gavaskar and Sachin Tendulkar: A Deep Dive into Batting Greatness

## Introduction

The debate over the greatest batsman in Indian cricket is as enduring as the game itself. This report delves into a comprehensive comparison of two cricketing legends: Sunil Gavaskar and Sachin Tendulkar. While Gavaskar paved the way with his technical precision and resilience against fearsome fast bowling in an era of uncovered pitches, Tendulkar redefined batting through his innovation and consistency over a record-breaking 24-year career. The research query, "Is Sunil Gavaskar a great batsman or Sachin a great batsman?" provides the framework to explore these contrasting yet parallel legacies.

## Historical Context and Era Differences

### Sunil Gavaskar’s Era

Sunil Gavaskar made his debut in 1971, a period marked by challenging pitch conditions, lack of modern protective gear, and the absence of advanced batting technology. The challenges of facing some of the world's most fearsome fast bowlers such as Dennis Lillee and Michael Holding on uncovered pitches created an environment where technical precision and mental fortitude were paramount. As the first cricketer to break the 10,000-run barrier in Test cricket, Gavaskar's career is noted for its pioneering spirit and technical ingenuity—for instance, adapting by batting left-handed in specific match scenarios to counter spin.

### Sachin Tendulkar’s Era

In contrast, Sachin Tendulkar's era, spanning from 1989 to 2013, witnessed an evolution in cricket strategy and technological advances. Better protective gear, refined playing surfaces, and enhanced coaching facilitated an environment where batting could be more aggressive and innovative. Tendulkar's adaptability saw him develop unique shots like the uppercut to counter short-pitched bowling, and his refined technique allowed him to overcome reverse swing and a plethora of bowler-friendly conditions. His record-breaking accumulation of runs in both Test and One Day formats speaks to his mastery in a rapidly changing cricketing landscape.

## Performance Metrics and Statistical Analysis

### Gavaskar’s Statistical Journey

- **Test Runs and Averages:** Over 125 Tests, Gavaskar accumulated 10,122 runs at an average of 51.12. His ability to consistently score runs against the formidable West Indies pace attacks is a testament to his technical discipline.
- **Centuries Against Adversities:** His 34 Test centuries, with 13 coming against the West Indies, illustrate his prowess under pressure and his ability to adapt to different match conditions.
- **Impactful Innings:** Notable innings, including his 220 in the West Indies and match-winning knocks in critical series, highlight his role as a trailblazer in establishing India’s presence on the international stage.

### Tendulkar’s Statistical Milestones

- **Record-Breaking Runs:** Tendulkar’s career is marked by a record 15,921 Test runs and 18,426 ODI runs, making him a prolific run-scorer in modern cricket.
- **Consistency Across Formats:** His ability to maintain high averages and score centuries consistently in both Tests and ODIs — highlighted by over 100 international centuries combined — manifests his technical brilliance and adaptability.
- **Adaptability and Innovation:** From developing the uppercut to his revolutionary methods in tackling reverse swing, Tendulkar's career is a repository of innovative cricketing techniques.

## Comparative Technical Analysis and Batting Styles

### Gavaskar’s Technique and Mental Toughness

Gavaskar’s batting technique was emblematic of classical correctness. His emphasis on balance and a methodical approach allowed him to play long, resilient innings. The challenges of his era meant that a small miscue could be fatal; hence, his technique was founded on a risk-averse yet highly effective strategy. His ability to adjust—such as switching his stance during particular domestic encounters—demonstrates an early form of strategic adaptability that later generations would build upon.

### Tendulkar’s Modern Craftsmanship

Tendulkar, known as the "God of Cricket," combined classical stroke play with modern innovation. His signature straight drive and the development of his uppercut were emblematic of a player who not only respected tradition but also pushed the boundaries of batting artistry. His practice routines—such as simulating reverse swing scenarios—exemplify a forward-thinking approach to training. Tendulkar’s style was often characterized by his aggressive yet precise shot selection, ensuring scoring even in the face of modern, sophisticated bowling strategies.

## Impact on Team Strategy and Match Outcomes

### Gavaskar’s Role in Team Success

As an opener, Gavaskar had the critical task of laying the foundation for his team, often facing hostile environments with minimal support. His contributions in pivotal series, such as India’s first Test series win in the Caribbean, bolstered the confidence of Indian cricket on the international stage. His technical expertise allowed him to anchor innings and set up platforms from which India could build competitive totals.

### Tendulkar’s All-Round Impact

Tendulkar’s longevity and flexibility allowed him to be a cornerstone in both team strategy and match outcomes. Whether it was stabilizing the innings in difficult encounters or accelerating during run chases, his role transcended mere statistics. His performance in match-critical scenarios, such as the iconic double century in an ODI or his resilient 241 against Australia, influenced how cricket teams structured their batting orders and game strategies.

## Subjective Evaluations and Expert Opinions

Expert opinions add a rich layer to the analysis. Former captains and peer players have often weighed in on the relative merits of both legends. For instance, West Indian great Sir Vivian Richards and Pakistani legend Imran Khan have both articulated their admiration for Gavaskar’s ability to thrive under extreme pressure. At the same time, renowned figures like Ross Taylor and Shane Bond have celebrated Tendulkar's all-format mastery and his mental toughness. These testimonies underscore that while statistical analysis can provide clarity on performance, the impact of these players is also measured by the subjective respect and admiration they command in the cricketing fraternity.

## Legacy and Influence on Future Generations

### Enduring Impact of Gavaskar

Sunil Gavaskar is not just remembered for his records but for setting a benchmark for technical soundness and resilience. His pioneering efforts against aggressive bowling attacks during challenging eras have inspired generations of openers. The techniques he honed—such as soft hands against spin and intelligent shot selection—are still studied by aspiring cricketers today. His influence can be seen in the way modern Indian openers approach the game, balancing caution with the necessity to score quickly.

### Tendulkar’s Global Outreach

In contrast, Sachin Tendulkar’s career transcends mere numbers. Known as the 'Little Master' by some and the 'God of Cricket' by many, his ability to adapt has influenced modern batting philosophies globally. His practices, mindset, and record-breaking consistency have helped define modern cricket's technical and strategic aspects. Tendulkar’s legacy is observed in how cricket academies shape the early training of young cricketers, emphasizing innovation blended with classical technique.

## Comparative Summary and Conclusions

The debate between whether Sunil Gavaskar or Sachin Tendulkar is the greater batsman does not lend itself to a definitive answer. Both have contributed uniquely to cricket. Gavaskar’s genius lay in his technical precision and resilience under severe conditions, qualities that made him a pioneer in a challenging era. Tendulkar’s brilliance, however, is underscored by his adaptability, innovation, and unparalleled statistical output in an era rich with advanced facilities and support structures.

While Gavaskar set the stage with groundbreaking achievements that inspired future legends, Tendulkar took those foundations to unimaginable heights by scoring runs that redefined benchmarks in both Test and ODI cricket. The comparative analysis thus illustrates that greatness in cricket can be defined in multiple dimensions—technical mastery, resilience, adaptability, and the ability to inspire future generations.

## Recommendations for Further Research

In light of the analysis, it would be valuable to explore further how changes in cricketing conditions and technological advancements have influenced batting strategies across eras. Additionally, studies involving advanced performance metrics could yield deeper insights into how modern techniques in coaching and training might integrate the timeless principles exhibited by both Gavaskar and Tendulkar.

## Conclusion

In conclusion, the legacies of Sunil Gavaskar and Sachin Tendulkar provide compelling evidence that greatness in cricket is multifaceted. Gavaskar’s technical mastery and fortitude under extreme conditions laid the groundwork for future generations, while Tendulkar’s innovative adaptations and record-breaking consistency transformed modern batting. This report shines a light on the evolution of cricket, illustrating that the debate is not about choosing one over the other but about appreciating the distinct contributions that have enriched Indian and global cricket.

*End of Report*


=====FOLLOW UP QUESTIONS=====


1. How have advances in training and technology altered the techniques and strategies of modern batsmen compared to the eras of Gavaskar and Tendulkar?
2. In what ways can further analysis of situational performance metrics help quantify the resilience and adaptability of batsmen from different eras?
3. How have team dynamics and support systems evolved to complement individual batting brilliance, and what lessons can be learned from historical approaches?
4. What role do psychological factors play in shaping a batsman's ability to adapt to changing match conditions, and how do these compare between Gavaskar and Tendulkar?
5. How can modern coaching practices integrate the technical and mental strategies of both Gavaskar and Tendulkar to develop well-rounded batsmen for the future?
